# Figure 4 — Historical Leiden–Claude replay

This is the canonical Figure 4 reproduction entry point. It reconstructs the existing `leiden__claude` cell predictions by joining the checksum-pinned 300-cluster Claude map to the exact frozen Leiden assignments. No model calls, fitting or clustering are performed.

The old GPT-named notebook is a compatibility redirect to this notebook. The original GPT reconstruction is retained only in the author-side change backup.


## Frozen inputs and interpretation

The [replay contract](../../configs/figure_04_historical_claude.json) pins all input hashes. The Claude labels come from the existing [cluster-count table](../../data/frozen/leiden_claude_diagnostic_identity/cluster_counts.csv), verified previously against the archived `leiden__claude` column. The [count audit](../../docs/leiden_claude_diagnostic_identity_audit.json) preserves the original cell-source identities and historical harmonization.

The reference H5AD must already be available locally at `data/raw/duke_research_repository/20260130_HuBMAP_experted_annotated.h5ad`, or set `FIGURE4_H5AD_PATH` to that exact file. Its SHA-256 is checked before use. It is distributed in `CODEX_annotated.zip` from [Duke record 505](https://research.repository.duke.edu/record/505). This notebook does not download data or access a model API.

Historical harmonization includes Paneth to Neuroendocrine, M1 to M2 macrophage and NK to CD7+ Immune. The original-28 comparison reuses the same predictions and is only a vocabulary-mismatch diagnostic. Eight regions belong to one donor. These results measure reference agreement and do not independently establish biological correctness.


In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import display
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")

configured = os.environ.get("SOURCE_REBUILD_REPO_ROOT")
candidates = [Path(configured).expanduser()] if configured else [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p.resolve() for p in candidates if (p / "configs/figure_04_historical_claude.json").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Run inside the repository or set SOURCE_REBUILD_REPO_ROOT to its root")
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from replay_figure_04_historical_claude import load_replay, render_panels, export


## 1. Verify the frozen Claude map and exact cell-key joins

Every frozen cluster is checked under both reference vocabularies. Expected historical counts are 123,099 direct matches, 63,067 cells in the within-cluster bound gap and 33,916 in the net annotation gap, including 7,486 minority-label matches.


In [ ]:
import json
REPLAY = load_replay(REPO_ROOT, os.environ.get("FIGURE4_H5AD_PATH"))
print(json.dumps(REPLAY["receipt"], indent=2))


## 2. Rebuild numerical panels A–E

A shows the frozen Leiden–Claude workflow. B shows the row-normalized confusion matrix. C shows pooled and region-specific cell-type F1. D uses the three previously selected historical cluster examples. E shows the cell-count-weighted aggregate.

For each cluster, A denotes direct reference agreement and U the reference-majority fraction. The three components are Direct reference agreement (A), Net annotation gap (U − A) and Within-cluster bound gap (1 − U). All direct matches, including minority-label matches, are retained. These quantities satisfy A + (U − A) + (1 − U) = 1 and are not mutually exclusive causal error categories.

The panels below redraw the frozen numerical results. The exact publication layout is the separate [versioned output preview](previews/figure_04.png). It is not loaded as a numerical input. The reconstruction has five manuscript panels, A–E.


In [ ]:
PANEL_FIGURES = render_panels(REPLAY)
import matplotlib.pyplot as plt
for figure in PANEL_FIGURES.values():
    plt.close(figure)


### Panel A


In [ ]:
display(PANEL_FIGURES["A"])


### Panel B


In [ ]:
display(PANEL_FIGURES["B"])


### Panel C


In [ ]:
display(PANEL_FIGURES["C"])


### Panel D


In [ ]:
display(PANEL_FIGURES["D"])


### Panel E


In [ ]:
display(PANEL_FIGURES["E"])


## 3. Export the replay and provenance

Generated PNG panels, a multipage PDF, the frozen cluster map and a source/output hash receipt are saved separately from the publication artwork.


In [ ]:
OUTPUT_ROOT = Path(os.environ.get("FIGURE4_OUTPUT_ROOT", str(REPO_ROOT / "outputs/source_rebuilt/figure_04_historical_claude")))
RECEIPT_PATH = export(REPLAY, PANEL_FIGURES, OUTPUT_ROOT)
print(RECEIPT_PATH)
